# [INFO] Glu-Stock: 03_EXECUTION_MONITOR
**Phase**: Institutional Recap & Execution | v18.25 (Consolidated)

This notebook manages finalized signals audited by the LLM Brain and handles portfolio monitoring.

In [ ]:
!pip install -q yfinance firebase-admin pandas pyTelegramBotAPI ta psutil python-dotenv

In [ ]:
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings, psutil, time
from firebase_admin import credentials, firestore
from datetime import datetime, timedelta
import telebot, ta
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw_firebase = user_secrets.get_secret("FIREBASE_KEY_JSON")
                token = user_secrets.get_secret("TELEGRAM_TOKEN")
                chat_id = user_secrets.get_secret("TELEGRAM_CHAT_ID")
                return {"key": json.loads(raw_firebase), "token": token, "chat_id": chat_id}
            except Exception: return {"key": None, "token": None, "chat_id": None}
        return {}

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()

    def wait_for_queue(self, queue_name: str, max_retries=5, interval=60):
        for i in range(max_retries):
            docs = self.db.collection(f"glu_stock_{queue_name}").get()
            if docs:
                tasks = []
                for doc in docs:
                    dt = doc.to_dict()
                    tasks.append(dt if 'ticker' in dt else dt.get('payload'))
                    doc.reference.delete()
                return tasks
            time.sleep(interval)
        return []

def run_institutional_cycle():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    finalized = fb.wait_for_queue('final_signals')
    report = f"**[GLU-STOCK] INSTITUTIONAL HUB**\nTargets: {len(finalized) if finalized else 0}"
    if secrets.get('token'):
        telebot.TeleBot(secrets['token']).send_message(secrets['chat_id'], report, parse_mode="Markdown")

run_institutional_cycle()